In [ ]:
# !pip install --upgrade kaleido numpy pandas neurokit2 plotly seaborn ts2vg pyxdf opencv-python seaborn ipywidgets pyarrow fastparquet

### Imports and helper functios

In [ ]:
from util import *

In [ ]:
# Adapted from Neurokit's read_xdf() to work with the data from this experiment
# https://neuropsychology.github.io/NeuroKit/functions/data.html#neurokit2.data.read_xdf
def read_xdf(subject: str, upsample=1, fillmissing=None):
    """**Read and tidy an XDF file**"""

    def get_markers(markers_stream):
        markers = markers_stream['time_series']
        assert all(len(marker) == 1 for marker in markers), 'Warning: There is an event containing more than one marker'
        markers = [marker[0] for marker in markers]

        designs = ["fair", "dark"]
        dark_first = json.load(open(m(subject, '*meta.json')[0]))['darkFirst']
        if dark_first:
            designs = designs[::-1]
        events = [
            "app/start", *[f"{s}/{t}" for s in stimuli for t in ("start", "end")],
 "app/end"
        ]
        # print(events)
        expected_marker_order = [f"{d}/{e}" for d in designs for e in events]

        # Handle missing fair/notification/end by inserting it at next routeChange
        markers_copy = markers.copy()
        timestamps_copy = list(markers_stream['time_stamps'])  # Convert numpy array to list

        # Find fair/notification/start and check if fair/notification/end is missing
        fair_notif_start_idx = next(i for i, marker in enumerate(markers_copy) if "fair/notification/start" in marker)
        fair_notif_end_exists = any("fair/notification/end" in marker for marker in markers_copy)

        # If fair/notification/end doesn't exist, insert it at next routeChange
        if not fair_notif_end_exists:
            # Look for next routeChange after fair/notification/start
            for i in range(fair_notif_start_idx + 1, len(markers_copy)):
                if re.search(r"routeChange/(flight|hotel)", markers_copy[i]):
                    # Insert fair/notification/end at this position
                    markers_copy.insert(i, "/".join(markers_copy[i].split("/", 3)[:3]) + "/fair/notification/end")
                    timestamps_copy.insert(i, timestamps_copy[i])  # Use same timestamp as routeChange
                    break

        # Only keep unique markers occurring in expected order
        relevant_indices = []
        previous_index = 0
        for marker in expected_marker_order:
            if any(marker in item for item in markers_copy[previous_index:]):
                index = next((i for i in range(previous_index, len(markers_copy)) if marker in markers_copy[i]), None)
                if index is None:
                    raise ValueError(f"No element containing '{marker}' found in markers from index {previous_index}")
                relevant_indices.append(index)
                previous_index = index
            # else:
            #     print(f'Missing marker: {marker}')

        timestamps = itemgetter(*relevant_indices)(timestamps_copy)
        markers = itemgetter(*relevant_indices)(markers_copy)

        return markers, timestamps

    try:
        import pyxdf
    except ImportError:
        raise ImportError(
            "The 'pyxdf' module is required for this function to run. ",
            "Please install it first (`pip install pyxdf`).",
        )

    # Load file
    print(f"Reading xdf file for subject: {subject}")
    streams, header = pyxdf.load_xdf(m(subject, '*.xdf')[0])

    # Remove any empty streams
    streams = [stream for stream in streams if len(stream['time_series'])]

    # Process markers stream first
    markers_stream = next(filter(lambda stream: isinstance(stream['time_series'], list), streams))
    streams.remove(markers_stream)
    markers, timestamps = get_markers(markers_stream)

    # print(f"Markers: {markers}")
    # print(f"Timestamps: {timestamps}")

    # Get the time range for analysis (from first to last marker)
    min_marker_time = min(timestamps)
    max_marker_time = max(timestamps)

    # Find the actual data range across all streams
    all_stream_times = []
    for stream in streams:
        all_stream_times.extend(stream["time_stamps"])

    data_start_time = min(all_stream_times)
    data_end_time = max(all_stream_times)

    # print(f"Data time range: {data_start_time} to {data_end_time}")
    # print(f"Marker time range: {min_marker_time} to {max_marker_time}")

    # Use the overlap between data and markers as the analysis window
    analysis_start = max(data_start_time, min_marker_time + epoch_start)
    analysis_end = min(data_end_time, max_marker_time + epoch_end)

    # print(f"Analysis window: {analysis_start} to {analysis_end}")

    # Set offset to analysis start
    offset = analysis_start

    markers_df = pd.DataFrame(markers, columns=['marker'])
    markers_df["marker"] = markers_df["marker"].str.split("/").str[2:6].apply("/".join)
    markers_df.index = pd.to_datetime(timestamps - offset, unit="s")

    # Process other streams and convert to dataframes
    dfs = []
    for stream in streams:
        time_mask = (stream["time_stamps"] >= analysis_start) & (stream["time_stamps"] <= analysis_end)

        if not np.any(time_mask):
            print(f"Warning: No data in analysis window for stream")
            continue

        filtered_timestamps = stream["time_stamps"][time_mask]
        filtered_data = stream["time_series"][time_mask]

        print(f"Stream data after filtering: {len(filtered_timestamps)} samples")
        # print(f"Time range: {filtered_timestamps.min() - offset} to {filtered_timestamps.max() - offset}")

        channels_info = stream["info"]["desc"][0]["channels"][0]["channel"]
        cols = [channels_info[i]["label"][0] for i in range(len(channels_info))]
        dat = pd.DataFrame(filtered_data, columns=cols)

        # Apply offset to timestamps
        dat.index = pd.to_datetime(filtered_timestamps - offset, unit="s")
        dfs.append(dat)

    if not dfs:
        raise ValueError("No valid data found in analysis window")

    # print(f"Number of data streams: {len(dfs)}")

    # Store info of each stream
    info = {
        "sampling_rates_original": [float(s["info"]["nominal_srate"][0]) for s in streams],
        "sampling_rates_effective": [float(s["info"]["effective_srate"]) for s in streams],
        "datetime": header["info"]["datetime"][0],
        "data": dfs,
        "subject": subject,
    }

    # Merge all dataframes by timestamps
    streams_df = dfs[0]
    for i in range(1, len(dfs)):
        streams_df = pd.merge(streams_df, dfs[i], how="outer", left_index=True, right_index=True)
    streams_df = streams_df.sort_index()

    # print(f"Merged dataframe shape: {streams_df.shape}")
    # print(f"Time range: {streams_df.index.min()} to {streams_df.index.max()}")

    # Resample and Interpolate
    info["sampling_rate"] = int(np.max(info["sampling_rates_original"]) * upsample)
    print(f"Target sampling rate: {info['sampling_rate']} Hz")

    if fillmissing is not None:
        fillmissing = int(info["sampling_rate"] * fillmissing)

    # Create new index with evenly spaced timestamps
    idx = pd.date_range(
        streams_df.index.min(),
        streams_df.index.max(),
        freq=str(1000 / info["sampling_rate"]) + "ms"
    )

    # print(f"New index length: {len(idx)}")

    # Reindex and interpolate
    streams_df = streams_df.reindex(streams_df.index.union(idx))

    # Only interpolate numeric columns
    numeric_cols = streams_df.select_dtypes(include=[np.number]).columns
    streams_df[numeric_cols] = streams_df[numeric_cols].interpolate(method="time", limit=fillmissing)

    # Use the new evenly spaced index
    streams_df = streams_df.reindex(idx)

    # Final data validation
    # print(f"Final dataframe shape: {streams_df.shape}")
    # print(f"NaN counts by column:")
    # for col in streams_df.columns:
    #     nan_count = streams_df[col].isna().sum()
    #     if nan_count > 0:
    #         print(f"  {col}: {nan_count} NaNs ({nan_count / len(streams_df) * 100:.1f}%)")

    return streams_df, markers_df, info


In [ ]:
def get_signals_and_events(i, streams_df, markers_df, info, plot=True):
    # Clean up the streams
    for col in ['RAW', 'RAW0']:
        if col in streams_df.columns:
            streams_df = streams_df.rename(columns={col: 'EDA'})
            break

    if False:
        plot_channels(
            streams_df, markers_df, title='Raw Channel Data', hide_end_markers=True)
        plot_gantt(markers_df)

    # Process EDA
    eda_signals, eda_info = nk.eda_process(streams_df['EDA'], sampling_rate=info["sampling_rate"],
                                           method_cleaning="biosppy")

    # print(eda_info)
    if True:
        nk.eda_plot(eda_signals, eda_info)
        fig = plt.gcf()
        fig._suptitle.set_text(f"Sub. {i}: {fig._suptitle.get_text()}")
        plt.show()


    # Concatenate processed signals
    signals = pd.concat([eda_signals], axis=1)

    # Reindex markers with numeric index for further processing
    nearest_indices = streams_df.index.get_indexer(markers_df.index, method='nearest')
    markers_numindexed = markers_df.copy()
    markers_numindexed.index = nearest_indices  # Use integer sample numbers

    if False:
        # Plot processed signals (only some columns)
        columns_to_plot = ['EDA_Tonic', 'EDA_Phasic']
        plot_channels(signals[columns_to_plot], markers_numindexed, title='Processed Channel Data',
                      hide_end_markers=True)
        plot_gantt(markers_df)

    # Remove the app markers
    markers_to_remove = '|'.join(['/app'])
    # markers_df = markers_df[~markers_df.marker.str.contains(markers_to_remove)]
    markers_numindexed = markers_numindexed[~markers_numindexed.marker.str.contains(markers_to_remove)]

    # Create events dictionary from gantt data for event-related analysis
    gantt_data = markers_to_gantt(markers_numindexed)

    labels = [d['marker'] for d in gantt_data]

    sites, designs, stimuli = zip(*[d['marker'].split('/')[:3] for d in gantt_data])

    events = dict(
        onset=[d['start'] for d in gantt_data],
        duration=[d['duration'] for d in gantt_data],
        label=labels,
        sites=sites,
        designs=designs,
        stimuli=stimuli
    )

    return signals, events

In [ ]:
def is_fixated_static(row):
    try:
        x = row["Fixation point X [DACS px]"]
        y = row["Fixation point Y [DACS px]"]

        if pd.isna(x) or pd.isna(y):
            return False

        site, design, stimulus = map(row.get, ("site", "design", "stimulus"))

        boxes = static_coords[stimulus]
        for _, box in boxes.iterrows():
            # If "design" or "site" is specified, and it doesn't match, skip it
            if ("design" in box and pd.notna(box["design"]) and box["design"] != design) or \
               ("site" in box and pd.notna(box["site"]) and box["site"] != site):
                # print("skipping", box)
                continue

            if box["left"] <= x <= box["right"] and box["top"] <= y <= box["bottom"]:
                return True  # Fixation is inside this box

        return False  # Not inside any valid box

    except Exception as e:
        print(f"Fixation check (static) failed: {e}")
        return False

def is_fixated_dynamic(row):
    try:
        x = row["Fixation point X [DACS px]"]
        y = row["Fixation point Y [DACS px]"]

        if pd.isna(x) or pd.isna(y) or pd.isna(row["left"]) or pd.isna(row["right"]) or pd.isna(row["top"]) or pd.isna(row["bottom"]):
            return False

        is_fixated = row["left"] <= x <= row["right"] and row["top"] <= y <= row["bottom"]
        return is_fixated
    except Exception as e:
        print(f"Fixation check (dynamic) failed: {e}")
        return False

In [ ]:
def get_epoch_features(subject, signals, events, info, plot=True):
    print("Getting epoch features for subject", subject)

    # Cache sampling rate and create events copy early
    sampling_rate = info["sampling_rate"]
    shifted_events = copy.deepcopy(events)

    # Build epochs from events
    static_epoch_signals_dict = nk.epochs_create(signals, events, sampling_rate=sampling_rate, epochs_start=epoch_start, epochs_end=epoch_end)
    eye_epoch_signals_dict = nk.epochs_create(signals, events, sampling_rate=sampling_rate)

    # Vectorized non-epoch signals creation
    # epoch_onsets = events['onset']
    # epoch_lengths = np.array([len(e) for e in static_epoch_signals_dict.values()])

    # Vectorized index creation
    # all_indices = []
    # for start, length in zip(epoch_onsets, epoch_lengths):
    #     all_indices.extend(range(start, start + length))

    # non_epoch_signals = signals.drop(index=all_indices, errors='ignore')

    if False:
        for epoch in epoch_signals_dict.values():
            plot_epoch(epoch, subplots=True)
            fig = plt.gcf()
            fig.suptitle(f"Sub. {subject}: Epoch from {epoch_start} to {epoch_end} seconds for Event: {epoch['Label'].values[0]}")
            fig.show()

    # Pre-calculate common values
    epoch_len = len(static_epoch_signals_dict[next(iter(static_epoch_signals_dict))])
    indie_epoch_lengths = [len(ep) for ep in eye_epoch_signals_dict.values()]

    # Vectorized dataframe creation
    static_epoch_signals = nk.epochs_to_df(static_epoch_signals_dict)
    static_epoch_signals = static_epoch_signals.assign(
        subject=subject,
        site=np.repeat(events['sites'], epoch_len),
        stimulus=np.repeat(events['stimuli'], epoch_len),
        design=np.repeat(events['designs'], epoch_len)
    )

    eye_epoch_signals = nk.epochs_to_df(eye_epoch_signals_dict)
    eye_epoch_signals = eye_epoch_signals.assign(
        subject=subject,
        site=np.repeat(events['sites'], indie_epoch_lengths),
        stimulus=np.repeat(events['stimuli'], indie_epoch_lengths),
        design=np.repeat(events['designs'], indie_epoch_lengths)
    )

    # Optimized time conversion and fixation setup
    eye_epoch_signals["Time"] = pd.to_timedelta(eye_epoch_signals["Time"], unit="s")
    eye_epoch_signals["Fixated"] = False
    ttff_dict = {}

    # Pre-filter and optimize fixation dataframe
    fixation_df_subject = fixation_df[fixation_df["subject"] == subject].copy()
    fixation_df_subject["timestamp_s"] = pd.to_timedelta(fixation_df_subject["Recording timestamp [ms]"] / 1000, unit="s")

    # Get unique labels once
    labels = eye_epoch_signals["Label"].unique()

    # Pre-filter eye_epoch_signals by label for faster lookups
    eye_signals_by_label = {label: eye_epoch_signals[eye_epoch_signals["Label"] == label].copy()
                           for label in labels}

    # Pre-calculate banner coords if needed
    banner_coords_cache = None

    for label in labels:
        # Get fixation start index more efficiently
        event_filter = fixation_df_subject["Event value"] == f"{label.split('/', maxsplit=1)[1]}/start"
        fixation_start_indices = fixation_df_subject[event_filter].index
        if len(fixation_start_indices) == 0:
            continue
        fixation_start_idx = fixation_start_indices[0]

        fixation_df_subject_filtered = fixation_df_subject.loc[fixation_start_idx:].copy()
        fixation_df_subject_filtered["Visible"] = True

        # Cache banner coords calculation for fair/notification labels
        if "fair/notification" in label:
            if banner_coords_cache is None:
                screen_rec_offset_cache = fixation_interval_df.loc[
                    (fixation_interval_df['subject'] == subject) &
                    (fixation_interval_df['TOI'].str.contains('screen')),
                    'Start_of_interval'
                ].values[0] / 1000

                banner_coords_cache = pd.read_csv("data_v2/banner_coords.csv")
                banner_coords_cache = banner_coords_cache.loc[banner_coords_cache['subject'] == subject].copy()
                banner_coords_cache['timestamp_relative'] -= screen_rec_offset_cache
                banner_coords_cache['timestamp_relative'] = pd.to_timedelta(banner_coords_cache['timestamp_relative'], unit='s')
                banner_coords_cache = banner_coords_cache.sort_values("timestamp_relative")

            # Pre-sorted merge
            fixation_df_subject_filtered = fixation_df_subject_filtered.sort_values("timestamp_s")
            fixation_df_subject_filtered = pd.merge_asof(
                fixation_df_subject_filtered,
                banner_coords_cache,
                left_on="timestamp_s",
                right_on="timestamp_relative",
                direction="nearest",
                tolerance=pd.Timedelta(seconds=1)
            )

            fixation_df_subject_filtered["Visible"] = ~fixation_df_subject_filtered[["left", "right", "top", "bottom"]].isna().any(axis=1)

        # Use pre-filtered data
        eye_epoch_signals_filtered = eye_signals_by_label[label]

        # Optimized time calculations
        zero_mask = eye_epoch_signals_filtered["Time"] >= pd.Timedelta(seconds=0)
        if zero_mask.any():
            zero_index = eye_epoch_signals_filtered.loc[zero_mask, "Time"].idxmin()
            start_time = eye_epoch_signals_filtered.loc[zero_index, "Time"]
            original_fixation_time = fixation_df_subject_filtered["timestamp_s"].iloc[0]
            shift = original_fixation_time - start_time
            fixation_df_subject_filtered = fixation_df_subject_filtered.copy()
            fixation_df_subject_filtered["timestamp_s"] -= shift

            # Pre-sort for merge_asof
            eye_sorted = eye_epoch_signals_filtered.sort_values("Time")
            fix_sorted = fixation_df_subject_filtered.sort_values("timestamp_s")

            merged_df = pd.merge_asof(
                eye_sorted,
                fix_sorted,
                left_on="Time",
                right_on="timestamp_s",
                direction="nearest",
                tolerance=pd.Timedelta(seconds=0.1)
            )
            merged_df.index = eye_epoch_signals_filtered.index

            # Apply fixation function vectorized
            fixation_func = is_fixated_dynamic if "fair/notification" in label else is_fixated_static
            fixated_values = merged_df.apply(fixation_func, axis=1)
            eye_epoch_signals.loc[eye_epoch_signals_filtered.index, "Fixated"] = fixated_values
            eye_epoch_signals.loc[eye_epoch_signals_filtered.index, "Visible"] = merged_df["Visible"]

            # Optimized TTFF calculation
            fixated_mask = eye_epoch_signals.loc[eye_epoch_signals_filtered.index, "Fixated"]
            fixated_times = eye_epoch_signals.loc[eye_epoch_signals_filtered.index, "Time"][fixated_mask]

            ttff = fixated_times.iloc[0].total_seconds() if not fixated_times.empty else np.nan
            ttff_dict[label] = ttff

    # Vectorized event onset updates
    ttff_values = np.array([ttff_dict.get(label) for label in events["label"]])
    ttff_values = np.nan_to_num(ttff_values, nan=0.0)
    print(ttff_values)
    shifted_events["onset"] = events["onset"] + (ttff_values + eda_latency) * sampling_rate
    actual_duration = events["duration"] - ttff_values * sampling_rate
    shifted_events["actual_duration"] = actual_duration
    shifted_events["duration"] = np.minimum(actual_duration, feature_window_cutoff * sampling_rate)

    # print("initial events", events)
    # print("updated events", shifted_events)

    # Create EDA epochs with modified events
    eda_epoch_signals = nk.epochs_create(signals, shifted_events, sampling_rate=sampling_rate)

    # Analyze epochs and extract features
    epoch_features = nk.eda_analyze(eda_epoch_signals, sampling_rate=sampling_rate, method="event-related")
    epoch_features.drop(columns=[col for col in epoch_features.columns if (col.startswith('SCR_') and not col.endswith(('RiseTime', 'Height'))) or col in ["Event_Onset", "EDA_Peak_Amplitude"]], inplace=True)

    print("epoch features", epoch_features.columns)


    # Vectorized statistics calculation
    interesting_eda_cols = ['EDA_Tonic', 'EDA_Phasic']
    stats_list = []

    for event, epoch in eda_epoch_signals.items():
        stats = epoch[interesting_eda_cols].agg(['mean', 'std', 'max', 'min', 'median'])
        event_stats = {}

        for signal_type in interesting_eda_cols:
            for stat in stats.index:
                event_stats[f"{signal_type}_{stat.capitalize()}"] = stats.loc[stat, signal_type]

        event_idx = events['label'].index(event)
        event_stats['duration'] = samples_to_seconds(shifted_events['duration'][event_idx], sampling_rate)

        # Normalized AUC
        event_stats["EDA_Phasic_AUC"] = np.trapezoid(np.maximum(0, epoch['EDA_Phasic']), dx=1/sampling_rate) / event_stats['duration']

        event_stats["SCR_Peaks_Rate"] = np.sum(epoch["SCR_Peaks"]) / event_stats['duration']
        event_stats["SCR_Recovery_Rate"] = np.sum(epoch["SCR_Recovery"]) / event_stats['duration']

        stats_list.append((event, event_stats))

    # Batch update epoch_features
    for event, event_stats in stats_list:
        for col, val in event_stats.items():
            epoch_features.loc[event, col] = val


    # Vectorized feature mapping
    epoch_features["TTFF"] = epoch_features["Label"].map(ttff_dict)

    # Group operations for efficiency
    fixated_sum_grouped = eye_epoch_signals.groupby("Label")["Fixated"].sum()
    epoch_features["Fixated_Sum"] = epoch_features["Label"].map(fixated_sum_grouped)

    eye_epoch_signals_groups = eye_epoch_signals.groupby("Label")
    visible_sum_grouped = eye_epoch_signals_groups["Visible"].sum()

    # Compute ratio (fixation within visible time only)
    fixated_ratio = fixated_sum_grouped.div(visible_sum_grouped.replace(0, np.nan), fill_value=0).fillna(0)
    # Map back to epoch_features
    epoch_features["Fixated_Ratio"] = epoch_features["Label"].map(fixated_ratio)

    # Vectorized site/stimulus/design assignment
    epoch_features["site"] = pd.Series(events["sites"]).values
    epoch_features["stimulus"] = pd.Series(events["stimuli"]).values
    epoch_features["design"] = pd.Series(events["designs"]).values

    # Do interval analysis to get features about the non-epoch signals
    non_epoch_features = None
        # nk.bio_analyze(non_epoch_signals, sampling_rate=sampling_rate, method="interval")

    return static_epoch_signals, epoch_features, non_epoch_features

In [ ]:
def process_subject(subject_str: str, subject: int):
    streams_df, markers_df, info = read_xdf(subject_str, upsample=1)
    signals, events = get_signals_and_events(subject, streams_df, markers_df, info, plot=0)
    epoch_signal_df, epoch_features, non_epoch_features = get_epoch_features(subject, signals, events, info, plot=0)

    counterbalancing_df, demographics_df = [
        pd.DataFrame([json.load(open(m(subject_str, f'*{name}.json')[0]))])
        for name in ['meta', 'demographics']
    ]

    questionnaire_json = json.load(open(m(subject_str, f'*questionnaire.json')[0]))
    questionnaire_json = {k.split("_", 1)[1] if k.startswith(sites) else k: v for k, v in questionnaire_json.items()}
    questionnaire_df = pd.DataFrame([questionnaire_json])

    # Load decision files and create decisions dataframe
    decisions = []
    for decision_file in m(subject_str, '*decision.json'):
        data = json.load(open(decision_file))
        for key, value in data.items():
            design, stimulus = key.split('_')[0:2]
            decisions.append({
                'subject': subject,
                'design': design,
                'stimulus': stimulus,
                'granted': value if stimulus != "cookies" else not value, # accidentally flipped the mapping in the frontend
                'Label': f"{design}/{stimulus}",
            })
    decisions_df = pd.DataFrame(decisions)

    for df in [
        # non_epoch_features,
        epoch_features, counterbalancing_df, demographics_df, questionnaire_df]:
        df.insert(0, 'subject', subject)

    return (
        # non_epoch_features,
            epoch_features, epoch_signal_df, counterbalancing_df, demographics_df, questionnaire_df, decisions_df)

### Load and process all data

#### Create data_v2 and data_v3 folders from data

In [ ]:
def read_with_print(path, **kwargs):
    """Read CSV/TSV file with appropriate separator and print status."""
    if kwargs is None:
        kwargs = {}
    print(f"Reading {path}")
    return pd.read_csv(path, index_col=None if path.endswith(".tsv") else 0, sep="\t" if path.endswith(".tsv") else ",", **kwargs)


def copy_file(src, dst):
    """Copy file and print status."""
    print(f"Copying {src} → {dst}")
    shutil.copy2(src, dst)


def create_v2_data():
    subjects = sorted([d for d in os.listdir('data/main') if not d.startswith('.')])[:]

    """Create v2 data folder and process files."""
    os.makedirs(os.path.join(V2_OUTPUT_DIR, 'screen'), exist_ok=True)

    # Process banner coordinates
    banner_coords_file = f"{V2_OUTPUT_DIR}/banner_coords.csv"
    if not os.path.isfile(banner_coords_file):
        for subject in subjects:
            print(f"Generating banner coordinates for {subject}...")
            video_file = glob(f'data/screen/{subject}*.mp4')[0]
            process_video_template(video_file).to_csv(banner_coords_file, index=False)
    else:
        print(f"File {banner_coords_file} already exists, not recreating.")

    # Copy TSV files
    tsv_mappings = [
        (glob(f"data/tobii/fixation/data/*.tsv")[0], "fixation_data.tsv"),
        *[(f, f"fixation_metrics_{basis}-based.tsv")
          for f, basis in zip(sorted(glob(f"data/tobii/fixation/metrics/*.tsv")), ['event', 'interval'])]
    ]
    for src, dst in tsv_mappings:
        copy_file(src, os.path.join(V2_OUTPUT_DIR, dst))

    # Read TSV files
    global fixation_df, fixation_event_df, fixation_interval_df
    fixation_df, fixation_event_df, fixation_interval_df = map(
        lambda df: (
            df
            .rename(columns={c: "subject" for c in df.columns if c.strip().startswith("Participant")})
            .assign(subject=lambda df: df["subject"].astype(str).str.extract(r"(\d+)", expand=False).astype(int))
        ),
        map(read_with_print, V2_TSV_FILES.values())
    )

    # Process subjects and copy videos
    collectors = {key: [] for key in V2_CSV_FILES.keys()}
    for i, subject in enumerate(subjects, start=1):
        print("\nProcessing subject", subject, "with index", i)
        results = process_subject(subject, i)
        for key, result in zip(V2_CSV_FILES.keys(), results):
            collectors[key].append(result)

        video_src = glob(f"data/screen/{subject}*.mp4")[0]
        copy_file(video_src, os.path.join(V2_OUTPUT_DIR, 'screen', f"{subject}.mp4"))

    # Save CSV files
    for key, dfs in collectors.items():
        if dfs:
            pd.concat(dfs).to_csv(V2_CSV_FILES[key], index=key == 'epoch_signals')

# Check if v3 needs creation
if not all(map(os.path.isfile, V3_FILES.values())) or 2 <= FORCE_CREATE_V:
    # Check if v2 needs creation
    if not all(map(os.path.isfile, V2_CSV_FILES.values())) or FORCE_CREATE_V == 2:
        print("Creating data_v2 folder...")
        create_v2_data()
    else:
        print("Folder data_v2 already looks complete, not recreating it.")

    # Create v3 folder
    print("Creating data_v3 folder...")
    os.makedirs(V3_OUTPUT_DIR, exist_ok=True)
    for key, v2_file in V2_CSV_FILES.items():
        v3_file = V3_FILES[key]
        print(f"Converting {v2_file} → {v3_file}")
        df = pd.read_csv(v2_file)
        # print(key, "before saving to parquet:")
        # print(df)
        df = df.round(4)
        df.to_parquet(v3_file, index=True, compression="zstd")
else:
    print("Folder data_v3 already looks complete, not recreating it.")

In [ ]:
print("Creating features.csv...")
ef, sf, cf, demographics_df, qf, df = map(pd.read_parquet, V3_FILES.values())
ef.columns = [col.lower() for col in ef.columns]

# Questionnaire: General DDP questions
stimulus_to_ddp = {
    "cookies": ["interfaceInterference", "forcedAction"],
    "geolocation": ["interfaceInterference", "forcedAction"],
    "notification": ["interfaceInterference", "socialEngineering"],
    "travelProtection": ["interfaceInterference", "sneaking"],
    "newsletter": ["forcedAction"],
}
records = []
for subject_id, q_row in qf.iterrows():
    for stim, ddps in stimulus_to_ddp.items():
        fam_vals = [float(q_row[f"{ddp}_familiar"]) for ddp in ddps]
        bot_vals = [float(q_row[f"{ddp}_bothered"]) for ddp in ddps]
        records.append({
            "subject": q_row["subject"],
            "stimulus": stim,
            "ddp_familiar": sum(fam_vals) / len(fam_vals),
            "ddp_bothered": sum(bot_vals) / len(bot_vals)
        })
q_ddp = pd.DataFrame(records)
ef = ef.merge(q_ddp, on=["subject", "stimulus"], how="left")

# Questionnaire: Stimuli questions
q_stim = qf[["subject"] + [col for col in qf.columns if any(stim in col for stim in stimuli)]]
q_long = q_stim.melt(id_vars="subject", var_name="question", value_name="response")
q_long["design"] = q_long["question"].str.extract(r"^(dark|fair)")
q_long["stimulus"] = q_long["question"].str.extract(f"({'|'.join(stimuli)})")
q_long["measure"] = q_long["question"].str.extract(r"_(aware|bothered)$")
q_pivot = q_long.pivot_table(
    index=["subject", "design", "stimulus"],
    columns="measure",
    values="response",
    aggfunc="first"  # safe here
).reset_index()
q_pivot["aware"] = q_pivot["aware"].astype(int)
ef = ef.merge(
    q_pivot,
    on=["subject", "design", "stimulus"],
    how="left"
)

# Decisions
ef = ef.merge(
    df[["subject", "design", "stimulus", "granted"]],
    on=["subject", "design", "stimulus"],
    how="left"
)
ef["granted"] = ef["granted"].where(pd.notna(ef["granted"]), False).astype(int)


# Demographics
demographics_df["is_male"] = (demographics_df["gender"] == "Male").astype(int)

freq_order = [
    "Never",
    "A few times a year",
    "A few times a month",
    "A few times a week",
    "Every day",
]
freq_map = {v: i for i, v in enumerate(freq_order)}
demographics_df["eng_speaking_freq"] = demographics_df["englishSpeakingFrequency"].map(freq_map).astype(int)
demographics_df["eng_understanding_freq"] = demographics_df["englishUnderstandingFrequency"].map(freq_map).astype(int)

proficiency_order = [
    "Beginner (A1)",
    "Elementary (A2)",
    "Intermediate (B1)",
    "Upper-Intermediate (B2)",
    "Advanced (C1)",
    "Native Speaker (C2)",
]
prof_map = {v: i for i, v in enumerate(proficiency_order)}
demographics_df["eng_proficiency"] = demographics_df["englishProficiency"].map(prof_map).astype(int)

ef = ef.merge(demographics_df[["subject", "age", "is_male", "eng_speaking_freq", "eng_understanding_freq", "eng_proficiency"]], on="subject", how="left")


# Counterbalancing
cf["dark_first"] = cf["darkFirst"].astype(int)
cf["hotel_first"] = cf["hotelFirst"].astype(int)
ef = ef.merge(cf, on="subject", how="left")


# Finalise
ef["dark"] = (ef["design"] == "dark").astype(int)
ef["condition_type"] = ef["design"] + "_" + ef["stimulus"]
ef["was_fixated"] = ef["ttff"].notna().astype(int)

# ef["scr_peaks_rate"] = ef["scr_peaks_rate"].astype(int)
# ef["scr_recovery_rate"] = ef["scr_recovery_rate"].astype(int)
ef.drop(columns=["label", "design"], inplace=True)

cols = [
    "subject", "site", "condition_type", "dark", "stimulus", "duration", "eda_tonic_mean", "eda_tonic_std", "eda_tonic_max", "eda_tonic_min", "eda_tonic_median",
    "eda_phasic_mean", "eda_phasic_std", "eda_phasic_max", "eda_phasic_min", "eda_phasic_median", "eda_phasic_auc", 'eda_scr',
    "scr_peaks_rate", "scr_recovery_rate", "aware", "bothered", "granted", "ddp_familiar", "ddp_bothered", "was_fixated", "fixated_ratio",
    "age", "is_male", "eng_speaking_freq", "eng_understanding_freq", "eng_proficiency",
    "dark_first", "hotel_first"
]

ef = ef[[col for col in cols if col in ef.columns]]

ef.to_csv("features.csv", float_format="%.4f", index=False)

print("Done.")